# Pokémon Battle Prediction — FDS Kaggle Competition

## 1. Clone the source repository

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/lucamartone/fds-pokemon.git"
REPO_DIR = Path("/kaggle/working/fds-pokemon")
RUN_STRENGTH_RESEARCH = True

if "<username>" in REPO_URL:
    raise ValueError("Replace REPO_URL with your public GitHub repository URL.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise ValueError(f"{REPO_DIR} exists but is not a Git checkout.")
else:
    print(f"Reusing existing checkout: {REPO_DIR}")

REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True,
).strip()
print("Repository commit:", REPO_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))


## 2. Locate competition data and build features

The feature builder only uses the available battle observations, the supplied type chart, and species statistics already present in the official JSONL files.


In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import platform
from pathlib import Path
from pokemon.main import build_all, cross_validate, make_submission, make_model, MODEL_NAMES

train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected exactly one train.jsonl, found {len(train_files)}")

DATA_DIR = train_files[0].parent
for filename in ("test.jsonl", "type_chart_gen2.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {DATA_DIR}")

def file_sha256(path):
    checksum = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            checksum.update(chunk)
    return checksum.hexdigest()

packages = {
    "numpy": "numpy", "pandas": "pandas", "scikit-learn": "sklearn",
    "lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost",
}
manifest = {
    "repo_commit": REPO_COMMIT,
    "python": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "versions": {
        name: importlib.import_module(module).__version__
        for name, module in packages.items()
    },
    "installed_versions": {name: importlib.metadata.version(name) for name in packages},
    "input_sha256": {
        name: file_sha256(DATA_DIR / name)
        for name in ("train.jsonl", "test.jsonl", "type_chart_gen2.csv")
    },
    "model_parameters": json.loads(
        json.dumps({name: make_model(name).get_params() for name in MODEL_NAMES}),
        parse_constant=str,
    ),
    "validation": {"n_splits": 5, "shuffle": True, "random_state": 42, "threshold": 0.5},
}
print("Loaded library versions:", json.dumps(manifest["versions"], sort_keys=True))
if manifest["versions"] != manifest["installed_versions"]:
    raise RuntimeError("Loaded and installed library versions differ. Restart the kernel after installing dependencies.")

features = build_all(DATA_DIR)
print("Training features:", features["A"].shape)
print("Test features:", features["B"].shape)
manifest["feature_columns"] = list(features["A"].columns)
manifest["feature_schema_sha256"] = hashlib.sha256(
    json.dumps(manifest["feature_columns"]).encode("utf-8"),
).hexdigest()
MANIFEST_PATH = Path("/kaggle/working/run_manifest.json")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print("Run manifest:", MANIFEST_PATH)


## 3. Validation

We use stratified five-fold cross-validation with **nested target encoding**. In every fold, validation labels are excluded from training and encoding statistics. The values printed below are the measured OOF accuracies for this run, not Kaggle test scores.

The previously reported 69.79% ensemble result is historical and is not guaranteed in another environment. One Kaggle execution measured 69.50% for LightGBM, 69.46% for the two-model blend and 69.50% for the three-model blend. Dependencies have minimum-version constraints; actual library versions, repository commit and input checksums are saved in `run_manifest.json` so differences can be investigated. Debugger and notebook-export warnings do not invalidate a successful run.


In [ ]:
scores = cross_validate(features)
for name, accuracy in scores.items():
    print(f"{name}: {accuracy:.2%}")
manifest["baseline_cv"] = scores
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")


## 4. Submission 1 — LightGBM

A strongly regularized tree ensemble using team, resource, tactical, and cross-fitted target-encoding features. Its measured five-fold accuracy is printed below before generating the CSV.


In [ ]:
print(f"Measured OOF accuracy: {scores['sub1_lgbm']:.2%}")
submission_1 = make_submission(features, "sub1_lgbm", "/kaggle/working")
print(submission_1)


## 5. Submission 2 — LightGBM + XGBoost

An **equal 50/50 average** of the two probability predictions. XGBoost provides a different boosting implementation and regularization regime. Its measured five-fold accuracy is printed below.


In [ ]:
print(f"Measured OOF accuracy: {scores['sub2_lgb_xgb']:.2%}")
submission_2 = make_submission(features, "sub2_lgb_xgb", "/kaggle/working")
print(submission_2)


## 6. Submission 3 — LightGBM + XGBoost + CatBoost

A **45% / 45% / 10% probability blend**, previously selected using labeled-training cross-validation. Its measured accuracy is printed below; averaging models does not guarantee improvement over LightGBM in every environment.


In [ ]:
print(f"Measured OOF accuracy: {scores['sub3_lgb_xgb_cat']:.2%}")
submission_3 = make_submission(features, "sub3_lgb_xgb_cat", "/kaggle/working")
print(submission_3)


## 7. Validate submission format

All predictions must have the same `battle_id` values and order as the official competition template, with `player_won` restricted to 0 or 1.


In [ ]:
import pandas as pd

sample_files = list(Path("/kaggle/input").rglob("sample_submission.csv"))
if len(sample_files) != 1:
    raise FileNotFoundError("Expected exactly one sample_submission.csv")

sample = pd.read_csv(sample_files[0])
for output in (submission_1, submission_2, submission_3):
    submission = pd.read_csv(output)
    assert submission.columns.tolist() == sample.columns.tolist()
    assert submission["battle_id"].equals(sample["battle_id"])
    assert submission["player_won"].isin([0, 1]).all()
    print(f"Validated {Path(output).name}: {len(submission)} predictions")


## 8. Active-strength candidate: validate in this Kaggle environment

The 637-column baseline above does not include the newer 42 interactions between active HP, stats, stat stages and status conditions. This experiment evaluates XGBoost with those features and seed 42, using the same five folds and nested encodings. It compares the candidate against the original ensemble **in this environment**, records OOF predictions and generates separate candidate CSVs.

The individual candidate previously scored 69.85% locally; that result is not a promised Kaggle score and does not establish 70% accuracy. `RUN_STRENGTH_RESEARCH = True` in the setup cell enables this additional experiment. Set it to `False` to run only the original three submissions. Candidate files require a separate selection decision; producing them does not select additional Kaggle submissions.


In [ ]:
if RUN_STRENGTH_RESEARCH:
    from strength_experiment import validate_strength, generate_candidates
    from pokemon.strength_features import build_strength_features

    strength_width = features["A"].shape[1] + build_strength_features(features["A"]).shape[1]
    print("Strength input columns (before target encodings):", strength_width)
    strength_results, strength_oof = validate_strength(features, seeds=(42,))
    strength_results["train_sha256"] = manifest["input_sha256"]["train.jsonl"]
    strength_results["repo_commit"] = REPO_COMMIT
    strength_results["run_manifest"] = str(MANIFEST_PATH)
    oof_path = Path("/kaggle/working/strength_oof.csv.gz")
    strength_oof.to_csv(oof_path, index=False)
    strength_results["oof_file"] = oof_path.name

    for name, result in strength_results["candidates"].items():
        print(f"{name}: {result['accuracy']:.2%}")
    candidate_paths = generate_candidates(
        features, "/kaggle/working/strength_candidates", seeds=(42,),
    )
    for output in candidate_paths.values():
        candidate = pd.read_csv(output)
        assert candidate.columns.tolist() == sample.columns.tolist()
        assert candidate["battle_id"].equals(sample["battle_id"])
        assert candidate["player_won"].isin([0, 1]).all()
        print(f"Validated {output}: {len(candidate)} predictions")

    strength_results["submission_files"] = candidate_paths
    Path("/kaggle/working/strength_results.json").write_text(
        json.dumps(strength_results, indent=2) + "\n", encoding="utf-8",
    )
    manifest["strength_cv"] = strength_results["candidates"]
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
else:
    print("Strength research disabled; set RUN_STRENGTH_RESEARCH = True in the setup cell to enable it.")
